# 杠杆 ETF 论文策略复刻（v1）

以 qlib `D.features` 为数据入口，按 close(t) 决策、下一交易日 open 执行，复刻趋势切换、滞后方差缩放、LETF-CPPI 和动态复制机制。outer 2023–2026 只评估、不选参。

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, ROOT.as_posix())
from src.workflow.leveraged_etf_paper_replay import run_leveraged_etf_paper_replay_card

summary = run_leveraged_etf_paper_replay_card(root=ROOT)
{
    'data_as_of': summary['data_audit']['as_of'],
    'promotion_decision': summary['promotion_decision'],
    'required_cost_pass_count': summary['required_cost_pass_count'],
    'recorder_id': summary['recorder_id'],
    'best_diagnostic': summary['selected_or_best_diagnostic']['strategy'],
}

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[99178:MainThread](2026-07-21 10:59:36,838) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[99178:MainThread](2026-07-21 10:59:37,260) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[99178:MainThread](2026-07-21 10:59:37,260) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[99178:MainThread](2026-07-21 10:59:37,871) INFO - qlib.workflow - [exp.py:258] - Experiment 345676235901942512 starts running ...


[99178:MainThread](2026-07-21 10:59:37,930) INFO - qlib.workflow - [recorder.py:345] - Recorder 68a5e9251dc64df3b4674aa669e32874 starts running under Experiment 345676235901942512 ...


[99178:MainThread](2026-07-21 10:59:38,059) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | waiting `async_log` Done


{'data_as_of': '2026-07-20',
 'promotion_decision': 'research_reject_no_paper_strategy_passed_all_gates',
 'required_cost_pass_count': 0,
 'recorder_id': '68a5e9251dc64df3b4674aa669e32874',
 'best_diagnostic': 'moreira_muir_lagged_inverse_variance'}

## 复刻边界

- Avellaneda–Zhang：1/3 SOXL + cash 的 1x 等效复制，比较日频与周频再平衡。
- Gayed：SOXX 收盘价高于 200 日均线时持有 2/3 SOXL（2x 等效），否则现金；另列 3x 压力版本。
- Moreira–Muir：下一月仓位与上一月 SOXX 已实现方差成反比；归一化常数只取 2011–2018。
- George–Trainor：年度 90% floor、multiplier=5/3、SOXL 上限 1/3、SOXX 相对上次再平衡移动 2.5% 触发调仓。
- Smart leverage：因缺 PIT T-bill/T-bond 与论文 bootstrap 神经控制，本卡不伪装成严格复刻。

In [2]:
from IPython.display import Markdown, display
display(Markdown((ROOT / 'research' / 'leveraged_etf_paper_replay_v1' / 'conclusion.md').read_text(encoding='utf-8')))

# 结论

## 跑了什么

使用 qlib 日线数据（截至 2026-07-20）按 close(t) 生成信号、在下一交易日开盘执行，复刻或适配了 200 日均线杠杆切换、滞后一月方差缩放、3x LETF CPPI，以及 1x 动态复制基准。所有策略均报告 0/10/25bp，outer 2023–2026 完全不参与参数选择。

## 关键结果

10bp 下通过全部正式门槛的策略数为 0。当前最佳诊断项为 `moreira_muir_lagged_inverse_variance`：CAGR 22.04%，MaxDD -20.79%，单日最差 -5.20%，CVaR5 -2.41%，真实滚动五日最差 -7.35%，SOXL 平均仓位 15.14%，上涨捕获 14.19%。

## 年份与成本差异

逐年 outer 结果见 `annual_outer_metrics.csv`，25bp 压力结果见 `strategy_comparison.csv`。2026 为截至数据日的未完整年度，不参与任何选择。

## 晋升决定

`research_reject_no_paper_strategy_passed_all_gates`。本卡是论文机制复盘，不晋升未通过全部风险、参与度、上涨捕获、双基准 Calmar 与 3/4 年度门槛的方案。

## 风险

CPPI 现金腿暂按 0 收益，缺少论文使用的 PIT 一年期国债收益；SOXX/SOXL 是半导体 ETF 映射，不等同论文的宽基指数；实际 SOXL 存在跳空、交易摩擦和杠杆衰减。`smart leverage` 的债券输入和 bootstrap 神经控制未在本卡伪复刻。

## 下一步

若继续，只应补齐 PIT T-bill/T-bond 序列后复刻 smart-leverage 数值框架，或把通过研究宽松门但未过正式门的论文规则作为模型的风险上限；不应再把底仓硬塞进不满足尾部门槛的模型。

不构成买卖建议。
